# Email features in eight versions: which drive clicks

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Orthogonal Arrays Testing (Taguchi Designs)** · a worked example from the StatNex teaching guide

**The story.** A marketing team tested four email features at once in only eight versions: subject line, send time, image and button colour. Each version went to a separate group of customers, and clicks per 1,000 emails were recorded.

**The question.** Which email features drive clicks?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `05_anova/orthogonal_array_analysis.csv` (8 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Experiment,A,B,C,D,Clicks per 1000 emails
1.0,1.0,1.0,1.0,1.0,20.0
2.0,1.0,1.0,2.0,2.0,22.0
3.0,1.0,2.0,1.0,2.0,23.0
4.0,1.0,2.0,2.0,1.0,24.0
5.0,2.0,1.0,1.0,2.0,27.0
6.0,2.0,1.0,2.0,1.0,26.0
7.0,2.0,2.0,1.0,1.0,31.0
8.0,2.0,2.0,2.0,2.0,33.0

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)

## The data

In [ ]:
# One measurement per run, in the order of the rows of the L8 array below.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
y = data["Clicks per 1000 emails"].to_numpy(float)

## The L8 orthogonal array

In [ ]:
# Eight runs, seven columns. Every column holds four 1s and four 2s, and every pair
# of columns is orthogonal, so seven effects can be estimated from eight runs.
L8 = np.array([
    [1, 1, 1, 1, 1, 1, 1],
    [1, 1, 1, 2, 2, 2, 2],
    [1, 2, 2, 1, 1, 2, 2],
    [1, 2, 2, 2, 2, 1, 1],
    [2, 1, 2, 1, 2, 1, 2],
    [2, 1, 2, 2, 1, 2, 1],
    [2, 2, 1, 1, 2, 2, 1],
    [2, 2, 1, 2, 1, 1, 2],
])

# The assignment pattern says what each column carries: a letter is a factor, two
# letters their interaction, and e an empty column that measures the error.
pattern = "A B AB C AC e D".split()
# Your own names for factors A, B, C, D (comma-separated); a letter without one keeps it.
custom = [part.strip() for part in ("Subject line,Send time,Image,Button colour" or "").split(",") if part.strip()]
rename = dict(zip("ABCD", custom))


def display(effect):
    return " * ".join(rename.get(letter, letter) for letter in effect)

## Analysis of variance table

In [ ]:
# With four runs at each level, a column's sum of squares is its contrast -- the level-1
# total minus the level-2 total -- squared and divided by 8. The e columns are pooled
# into the error, one degree of freedom each. An e column is pure error only if no
# interaction lives in it, and with one or two error df the F test has little power
# (the 5% point of F(1, 1) is 161), so read the P values as rough. statsmodels' ols()
# and anova_lm() on factors built from the columns give the same table; the column
# contrasts are the textbook (Taguchi) computation and work for any assignment pattern.
ss = {}
error_ss, error_df = 0.0, 0
for column, effect in enumerate(pattern):
    contrast = y[L8[:, column] == 1].sum() - y[L8[:, column] == 2].sum()
    if effect == "e":
        error_ss += contrast**2 / 8
        error_df += 1
    else:
        ss[effect] = contrast**2 / 8
ms_error = error_ss / error_df

# Main effects first, then interactions. When the error columns hold no variation
# at all, F has nothing to divide by and is left blank.
order = sorted(ss, key=lambda effect: (len(effect) > 1, effect))
rows = []
for e in order:
    f_value = ss[e] / ms_error if ms_error > 0 else np.nan
    rows.append([ss[e], 1, ss[e], f_value, stats.f.sf(f_value, 1, error_df)])
rows.append([error_ss, error_df, ms_error, np.nan, np.nan])
rows.append([((y - y.mean()) ** 2).sum(), 7, np.nan, np.nan, np.nan])
anova = pd.DataFrame(rows, index=[display(e) for e in order] + ["Error", "Total"],
                     columns=["SS", "df", "MS", "F", "P"])
print(anova)

## Means at each level

In [ ]:
# The effect is the level-2 mean minus the level-1 mean: the slope of the factor's line.
rows = []
for column, effect in enumerate(pattern):
    if effect != "e" and len(effect) == 1:
        one, two = y[L8[:, column] == 1].mean(), y[L8[:, column] == 2].mean()
        rows.append([display(effect), one, two, two - one])
effects = pd.DataFrame(rows, columns=["factor", "mean at level 1", "mean at level 2", "effect"])
print()
print(effects.to_string(index=False))

## What the result says

An orthogonal array tests four features in only eight versions. The analysis of variance table gives each feature an F, its effect compared with the noise, and a P value, the chance of an effect at least this large if the feature did nothing.

Subject line has F = 196 and P = 0.045. In the means by level, statements earn 29.25 clicks per thousand emails and questions 22.25, an effect of 7. Send time has an effect of 4 with P = 0.079, which is not significant. Image and button colour each have P = 0.295.

The error row has only 1 degree of freedom, so the noise is measured from very little information. Use statements, treat evening sending as a maybe, and confirm the winner in a proper test.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.